# Media Authenticity Detection - Image Preprocessing

This notebook contains the complete preprocessing pipeline for the Media Authenticity Detection system. It handles:
- Recursive image discovery
- Face detection with fallbacks (MediaPipe and OpenCV Haar Cascades)
- Normalization (224x224 RGB)
- Stratified dataset splitting (Train/Val/Test)

In [1]:
import os
import glob
import cv2
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import logging
import uuid

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Constants
RAW_DATA_DIR = os.path.join("..", "data", "raw")
PROCESSED_DATA_DIR = os.path.join("..", "data", "processed", "images")
TARGET_SIZE = (224, 224)
SUPPORTED_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp')

## Face Detector Initialization
This function initializes the best available face detector. It favors MediaPipe but falls back to OpenCV Haar Cascades if MediaPipe is not properly configured on the system.

In [2]:
def initialize_face_detector():
    """Tries to initialize MediaPipe, falls back to OpenCV Haar Cascades."""
    # 1. Try MediaPipe (Legacy Solutions)
    try:
        import mediapipe as mp
        detector = mp.solutions.face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)
        logging.info("Using MediaPipe (Solutions API) for face detection.")
        return ("mediapipe", detector)
    except (ImportError, AttributeError):
        pass

    # 2. Try OpenCV Haar Cascade (Universal Fallback)
    try:
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        detector = cv2.CascadeClassifier(cascade_path)
        if not detector.empty():
            logging.info("Using OpenCV Haar Cascade for face detection.")
            return ("opencv", detector)
    except Exception:
        pass

    logging.warning("No face detector available. Will process full images only.")
    return ("none", None)

## Helper Functions
Handles recursive file discovery, image processing, and saving.

In [3]:
def get_image_files(directory):
    """Retrieve all supported image files in a directory (recursive, deduplicated)."""
    files = set()
    for ext in SUPPORTED_EXTENSIONS:
        # On Windows, glob is case-insensitive, but we use a set to be safe
        files.update(glob.glob(os.path.join(directory, "**", ext), recursive=True))
        files.update(glob.glob(os.path.join(directory, "**", ext.upper()), recursive=True))
    return sorted(list(files))

def process_image(img_path, detector_tuple, padding=0.2):
    """Reads, processes (face crop or keep full), and resizes the image."""
    detector_type, detector = detector_tuple
    
    img = cv2.imread(img_path)
    if img is None:
        return None
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape
    crop = None

    if detector_type == "mediapipe":
        results = detector.process(img_rgb)
        if results.detections:
            detection = results.detections[0] 
            bboxC = detection.location_data.relative_bounding_box
            xmin = int(bboxC.xmin * w)
            ymin = int(bboxC.ymin * h)
            box_w = int(bboxC.width * w)
            box_h = int(bboxC.height * h)
            crop = (xmin, ymin, box_w, box_h)
            
    elif detector_type == "opencv":
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        if len(faces) > 0:
            faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
            crop = faces[0]

    if crop is not None:
        x, y, bw, bh = crop
        pad_x = int(bw * padding)
        pad_y = int(bh * padding)
        
        xmin = max(0, x - pad_x)
        ymin = max(0, y - pad_y)
        xmax = min(w, x + bw + 2 * pad_x)
        ymax = min(h, y + bh + 2 * pad_y)
        
        processed_img = img_rgb[ymin:ymax, xmin:xmax]
        if processed_img.size == 0:
            processed_img = img_rgb
    else:
        processed_img = img_rgb
        
    processed_img = cv2.resize(processed_img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    return processed_img

def save_image(img_array, save_path):
    """Saves numpy array as an image file."""
    img_pil = Image.fromarray(img_array)
    img_pil.save(save_path, "JPEG", quality=95)

def create_directory_structure():
    """Returns a dictionary of output paths and auto-creates them."""
    splits = ['train', 'val', 'test']
    labels = ['real', 'fake']
    paths = {}
    for split in splits:
        paths[split] = {}
        for label in labels:
            path = os.path.join(PROCESSED_DATA_DIR, split, label)
            os.makedirs(path, exist_ok=True)
            paths[split][label] = path
    return paths

## Main Execution Loop
Runs the pipeline for 'real' and 'fake' classes.

In [4]:
def run_preprocessing():
    logging.info("Starting preprocessing pipeline...")
    detector_tuple = initialize_face_detector()
    dir_paths = create_directory_structure()
    
    for label in ['real', 'fake']:
        raw_target_dir = os.path.join(RAW_DATA_DIR, f"{label}_images")
        if not os.path.exists(raw_target_dir):
            logging.warning(f"Directory not found: {raw_target_dir}. Skipping label '{label}'.")
            continue
            
        img_files = get_image_files(raw_target_dir)
        if not img_files:
            logging.warning(f"No valid images found in {raw_target_dir}.")
            continue
        
        logging.info(f"Processing {len(img_files)} images for label '{label}'...")
        
        valid_paths = [p for p in img_files if cv2.imread(p) is not None]
        if not valid_paths:
            logging.warning(f"All images in {label}_images/ were unreadable.")
            continue
            
        train_paths, temp_paths = train_test_split(valid_paths, test_size=0.30, random_state=42)
        val_paths, test_paths = train_test_split(temp_paths, test_size=0.50, random_state=42)
        
        mapping = [('train', train_paths), ('val', val_paths), ('test', test_paths)]
        
        for split_inner, paths_inner in mapping:
            save_dir = dir_paths[split_inner][label]
            for path in tqdm(paths_inner, desc=f"Processing {split_inner} ({label})"):
                proc_img = process_image(path, detector_tuple)
                if proc_img is not None:
                    fname = os.path.splitext(os.path.basename(path))[0]
                    unique_name = f"{fname}_{uuid.uuid4().hex[:6]}.jpg"
                    save_image(proc_img, os.path.join(save_dir, unique_name))
        
        logging.info(f"Split Summary for '{label}': Train={len(train_paths)}, Val={len(val_paths)}, Test={len(test_paths)}")

    logging.info("Preprocessing complete.")

# Execute the pipeline
run_preprocessing()

INFO: Starting preprocessing pipeline...
INFO: Using OpenCV Haar Cascade for face detection.
INFO: Processing 50061 images for label 'real'...
Processing test (real): 100%|██████████| 7510/7510 [00:34<00:00, 215.67it/s]
INFO: Split Summary for 'real': Train=35042, Val=7509, Test=7510
INFO: Processing 50061 images for label 'fake'...
Processing test (fake): 100%|██████████| 7510/7510 [00:29<00:00, 258.22it/s]
INFO: Split Summary for 'fake': Train=35042, Val=7509, Test=7510
INFO: Preprocessing complete.
